# MODUL PRAKTIKUM BIG DATA
## Pertemuan 3 — Pengenalan Big Data & Instalasi Apache Hadoop (Arch Linux Safe Edition)

| Informasi | Keterangan |
|---|---|
| **Mata Kuliah** | Praktikum Big Data |
| **Program Studi** | Teknologi Informasi — Universitas Tidar |
| **Sistem Operasi** | Arch Linux / CachyOS (Lingkungan Terisolasi & Aman) |
| **Direktori Kerja** | `/mnt/alterra/repos/praktikum-big-data` (uv project & Git repo) |
| **Perangkat Lunak** | Apache Hadoop 3.4.3 & OpenJDK 11 (Terisolasi) |
| **Topik Pembahasan** | Konsep Big Data 5V, Ekosistem Hadoop, Instalasi Fish Shell PATH, Operasi HDFS, Studi Kasus E-Commerce |

---

### 🛡️ Jaminan Keamanan Sistem (Zero-Conflict Arch Linux)
Modul versi ini dirancang khusus agar Anda memahami **apa fungsi setiap baris perintah** dan **mengapa baris itu harus dijalankan**, sekaligus menjamin sistem Arch Linux Anda tetap aman:
1. **Java 21 Sistem & IntelliJ Tetap Utuh**: Perintah `archlinux-java` di sistemmu tetap menggunakan default bawaan (Java 21). Java 11 hanya dipanggil secara privat oleh Hadoop lewat konfigurasi internal `hadoop-env.sh`.
2. **Integrasi Fish Shell Modern**: Menggunakan `fish_add_path` bawaan CachyOS/Arch Linux sehingga biner Hadoop langsung dikenali di semua terminal dan notebook tanpa boilerplate panjang.
3. **Menggunakan Standar Variabel Linux (`$USER`)**: Sesuai konvensi Linux/Unix, path direktori kerja menggunakan `$USER` (otomatis memanggil username aktif, misal `/user/kaky/...`).

---
## 1. Landasan Teori: Mengapa Kita Butuh Hadoop?

### 1.1 Masalah pada Pendekatan Konvensional (Single-Machine)
Pada praktikum sebelumnya, kita menggunakan **Pandas** di Python. Pandas bekerja dengan cara memuat seluruh dataset langsung ke dalam memori RAM komputer. Pendekatan ini memiliki limitasi fatal: saat ukuran data melampaui kapasitas RAM laptop (misal data 50 GB di laptop dengan RAM 8 GB), program akan langsung *crash* dengan pesan `MemoryError`.

### 1.2 Kerangka Karakteristik 5V Big Data
1. **Volume (Ukuran)**: Jumlah data yang mencapai skala Terabyte (TB) hingga Petabyte (PB).
2. **Velocity (Kecepatan)**: Kecepatan data masuk secara kontinu (misal jutaan transaksi kasir per detik).
3. **Variety (Keragaman)**: Format data bercampur antara data tabel, log teks, file JSON, dan citra/video.
4. **Veracity (Keandalan)**: Ketidakpastian kualitas data mentah yang membutuhkan pembersihan (*filtering*).
5. **Value (Nilai Manfaat)**: Wawasan bisnis yang diekstraksi dari data setelah diproses.

### 1.3 Arsitektur Inti Ekosistem Apache Hadoop
Hadoop memecahkan masalah skala dengan mendistribusikan penyimpanan dan pemrosesan ke banyak komputer (*cluster*):
* **HDFS (Hadoop Distributed File System)**: Layer penyimpanan. Berkas besar dipecah menjadi blok-blok (standar 128 MB), lalu disebar ke berbagai node pekerja (**DataNode**) dengan koordinasi dari master (**NameNode**).
* **YARN (Yet Another Resource Negotiator)**: Layer manajemen sumber daya. Bertugas membagi jatah CPU dan RAM klaster ke aplikasi yang sedang berjalan via **ResourceManager** dan **NodeManager**.
* **MapReduce**: Model pemrosesan data paralel. Terdiri atas fase *Map* (memilah & memetakan data) dan fase *Reduce* (mengagregasi/merangkum hasil akhir).

---
## 2. Panduan Lengkap Instalasi Hadoop di Arch Linux (Langkah Terminal)

Jalankan langkah-langkah di bawah ini satu per satu di terminal Anda.

---

### Langkah A: Mengapa Perlu SSH dan Bagaimana Cara Kerjanya?

Hadoop aslinya dirancang untuk klaster multi-komputer. Di klaster nyata, komputer master (*NameNode*) menyalakan proses di komputer pekerja (*DataNode*) menggunakan protokol remote **SSH**.  
Pada mode *single-node* (laptop sendiri), skrip otomatis Hadoop (`start-dfs.sh` dan `start-yarn.sh`) **tetap menggunakan logika SSH yang sama** untuk menghubungi `localhost`.

```bash
# 1. Menginstal Java 11, OpenSSH, dan downloader Wget
sudo pacman -S jdk11-openjdk openssh wget

# 2. Membangunkan server SSH di laptop (daemon sshd secara on-demand)
sudo systemctl start sshd

# 3. Membuat sepasang kunci kriptografi (passphrase kosong agar tidak minta password)
ssh-keygen -t rsa -P '' -f ~/.ssh/id_rsa

# 4. Mendaftarkan kunci publik ke daftar kunci yang dipercaya
cat ~/.ssh/id_rsa.pub >> ~/.ssh/authorized_keys
chmod 0600 ~/.ssh/authorized_keys
chmod 0700 ~/.ssh

# 5. Uji coba koneksi SSH ke diri sendiri (ketik 'yes' saat pertama kali, lalu 'exit')
ssh localhost
exit
```

---

### Langkah B: Mengunduh dan Mengekstrak Apache Hadoop 3.4.3

```bash
cd ~
wget https://downloads.apache.org/hadoop/common/hadoop-3.4.3/hadoop-3.4.3.tar.gz
tar -xzvf hadoop-3.4.3.tar.gz
mv hadoop-3.4.3 hadoop
```

---

### Langkah C: Kunci Java 11 Khusus Hadoop di `hadoop-env.sh` (Kunci Isolasi)

Buka berkas konfigurasi internal environment milik Hadoop:
```bash
nano ~/hadoop/etc/hadoop/hadoop-env.sh
```
*(Atau buka via GNOME Text Editor lewat Nautilus).*  
Tambahkan baris berikut di baris paling bawah:
```bash
export JAVA_HOME=/usr/lib/jvm/java-11-openjdk
```

---

### Langkah D: Daftarkan Hadoop ke PATH Shell Laptop (Khusus Fish Shell)

Agar perintah `hdfs` dapat dipanggil langsung dari terminal maupun notebook manapun tanpa perlu menyalin kode inisialisasi berulang kali, daftarkan jalurnya menggunakan fitur resmi Fish Shell:

```bash
fish_add_path ~/hadoop/bin ~/hadoop/sbin
```
> **Catatan Keamanan:** Perintah ini **hanya** mendaftarkan lokasi program Hadoop ke `$PATH`. Java default laptopmu tetap aman di Java 21 untuk IntelliJ IDEA.

---

### Langkah E: Mengonfigurasi 4 Berkas XML Hadoop (Otak Sistem)

#### 1. `core-site.xml` — Menentukan Alamat Pusat HDFS
Buka `~/hadoop/etc/hadoop/core-site.xml`, ubah bagian `<configuration>` menjadi:
```xml
<configuration>
    <property>
        <name>fs.defaultFS</name>
        <value>hdfs://localhost:9000</value>
    </property>
</configuration>
```

#### 2. `hdfs-site.xml` — Pengaturan Replikasi dan Tempat Penyimpanan Fisik
Buat folder tempat penyimpanan data HDFS terlebih dahulu di laptop:
```bash
mkdir -p ~/hadoopdata/hdfs/namenode
mkdir -p ~/hadoopdata/hdfs/datanode
```
Lalu buka `~/hadoop/etc/hadoop/hdfs-site.xml` dan ubah isinya menjadi:
```xml
<configuration>
    <property>
        <name>dfs.replication</name>
        <value>1</value>
    </property>
    <property>
        <name>dfs.namenode.name.dir</name>
        <value>/home/kaky/hadoopdata/hdfs/namenode</value>
    </property>
    <property>
        <name>dfs.datanode.data.dir</name>
        <value>/home/kaky/hadoopdata/hdfs/datanode</value>
    </property>
</configuration>
```

#### 3. `mapred-site.xml` — Menentukan Mesin Eksekusi MapReduce
Buka `~/hadoop/etc/hadoop/mapred-site.xml`, ubah isinya menjadi:
```xml
<configuration>
    <property>
        <name>mapreduce.framework.name</name>
        <value>yarn</value>
    </property>
</configuration>
```

#### 4. `yarn-site.xml` — Pengaturan Shuffle Service
Buka `~/hadoop/etc/hadoop/yarn-site.xml`, ubah isinya menjadi:
```xml
<configuration>
    <property>
        <name>yarn.nodemanager.aux-services</name>
        <value>mapreduce_shuffle</value>
    </property>
</configuration>
```

---

### Langkah F: Format NameNode (Hanya 1x Seumur Hidup HDFS)

```bash
~/hadoop/bin/hdfs namenode -format
```
* **Tanda Sukses:** Perhatikan output terminal, pastikan terdapat baris: `Storage directory ... has been successfully formatted`.

---

### Langkah G: Menjalankan Daemon Hadoop & Verifikasi

Nyalakan HDFS dan YARN:
```bash
~/hadoop/sbin/start-dfs.sh
~/hadoop/sbin/start-yarn.sh
```

Verifikasi seluruh proses Java yang aktif:
```bash
jps
```
*(Harus muncul: NameNode, DataNode, SecondaryNameNode, ResourceManager, NodeManager, Jps).* 

---

### Langkah H: Menjalankan Jupyter Notebook dari Repositori Proyek
Arahkan ke folder repositori praktikum, lalu jalankan Jupyter via `uv`:
```bash
cd /mnt/alterra/repos/praktikum-big-data
uv run jupyter notebook
```

---
## 3. Verifikasi Koneksi Sesi Notebook

Karena kita sudah menambahkan `fish_add_path ~/hadoop/bin ~/hadoop/sbin`, seluruh perintah `!hdfs` di notebook ini dapat langsung dieksekusi secara instan!

In [ ]:
# 1. Memeriksa versi Hadoop yang terhubung
!hdfs version

print("\n--- PROSES DAEMON AKTIF (JPS) ---")
# 2. Memeriksa keberadaan NameNode, DataNode, SecondaryNameNode, ResourceManager, NodeManager
!jps

> 🌐 **Dashboard Visual Web UI:**
> Buka tab baru di browser Anda untuk melihat dashboard grafis bawaan Hadoop:
> * **NameNode Web UI:** [http://localhost:9870](http://localhost:9870) *(Melihat kesehatan disk HDFS, kapasitas blok, dan menu Utilities -> Browse the file system)*.
> * **YARN ResourceManager:** [http://localhost:8088](http://localhost:8088) *(Melihat daftar alokasi CPU, RAM, dan status job yang berjalan)*.

---
## 4. Hands-On: Perintah Dasar HDFS

Sistem berkas HDFS terpisah dari sistem berkas Linux lokal Anda.  
Pola umum sintaks HDFS adalah: **`hdfs dfs -[nama_perintah] [argumen]`**  
Kita menggunakan variabel lingkungan bawaan Linux **`$USER`** yang otomatis merujuk ke akun login Anda (misalnya `/user/kaky/...`).

### 4.1 Membuat Direktori di HDFS (`-mkdir`)
* **Fungsi**: Membuat folder baru di dalam sistem penyimpanan HDFS.
* **Opsi `-p`**: Memastikan folder induk otomatis dibuat jika belum ada.

In [ ]:
# Membuat folder /user/$USER/praktikum di HDFS
!hdfs dfs -mkdir -p /user/$USER/praktikum

# Memeriksa daftar isi direktori /user/$USER untuk pembuktian
!hdfs dfs -ls /user/$USER

### 4.2 Mengunggah Berkas dari Disk Lokal ke HDFS (`-put`)
* **Fungsi**: Menyalin (*copy*) berkas yang ada di hard disk lokal laptop Anda ke dalam sistem blok HDFS.
* **Opsi `-f`**: *Force overwrite*, menimpa berkas jika sudah pernah diunggah sebelumnya.

In [ ]:
# 1. Buat berkas teks contoh di disk lokal laptop
with open("contoh_lokal.txt", "w") as f:
    f.write("Halo dari Praktikum Big Data di Arch Linux!\n")
    f.write("Berkas ini dibuat di disk lokal lalu diunggah ke cluster HDFS.\n")

print("Berkas lokal 'contoh_lokal.txt' berhasil dibuat di folder repo.")

# 2. Unggah berkas tersebut ke direktori HDFS yang sudah dibuat tadi
!hdfs dfs -put -f contoh_lokal.txt /user/$USER/praktikum/

# 3. Verifikasi apakah berkas sudah terdaftar di HDFS
!hdfs dfs -ls /user/$USER/praktikum

### 4.3 Membaca Isi Berkas Langsung dari HDFS (`-cat`)
* **Fungsi**: Membaca dan mencetak isi berkas teks yang tersimpan di HDFS langsung ke layar output tanpa perlu menyalin atau mengunduhnya kembali ke disk lokal.

In [ ]:
!hdfs dfs -cat /user/$USER/praktikum/contoh_lokal.txt

### 4.4 Mengunduh Berkas dari HDFS ke Disk Lokal (`-get`)
* **Fungsi**: Kebalikan dari `put`. Mengambil berkas dari penyimpanan HDFS dan menyimpannya kembali ke sistem berkas lokal laptop Anda.

In [ ]:
# Mengunduh berkas dari HDFS dan menamainya 'hasil_unduh_dari_hdfs.txt' di lokal
!hdfs dfs -get -f /user/$USER/praktikum/contoh_lokal.txt hasil_unduh_dari_hdfs.txt

# Periksa isi berkas hasil unduhan lokal untuk memastikan isinya utuh
!cat hasil_unduh_dari_hdfs.txt

### 4.5 Memeriksa Kapasitas Ruang dan Jumlah Berkas (`-du` & `-count`)
* **`-du -h`**: *Disk Usage*, menampilkan ukuran berkas di HDFS dalam format human-readable (KB, MB).
* **`-count`**: Menghitung total folder, jumlah berkas, dan total bytes secara instan.

In [ ]:
# Cek ukuran berkas di HDFS
!hdfs dfs -du -h /user/$USER/praktikum

# Hitung jumlah folder dan berkas
!hdfs dfs -count /user/$USER/praktikum

### 4.6 Menghapus Berkas di HDFS (`-rm`)
* **Fungsi**: Menghapus berkas dari HDFS dan membebaskan ruang penyimpanan blok.

In [ ]:
# Hapus berkas contoh di HDFS
!hdfs dfs -rm /user/$USER/praktikum/contoh_lokal.txt

# Buktikan bahwa folder praktikum kini telah kosong
!hdfs dfs -ls /user/$USER/praktikum

### Tabel Ringkasan Sintaks Dasar HDFS

| Perintah | Fungsi Utama | Contoh Kasus Nyata |
|---|---|---|
| `hdfs dfs -mkdir -p [path]` | Membuat folder baru di HDFS | Menyiapkan folder `/user/$USER/data_transaksi` |
| `hdfs dfs -ls [path]` | Melihat daftar isi folder | Memeriksa apakah file sudah terunggah |
| `hdfs dfs -put [lokal] [hdfs]` | Mengunggah file dari lokal ke HDFS | Memindahkan file CSV log harian ke klaster |
| `hdfs dfs -get [hdfs] [lokal]` | Mengunduh file dari HDFS ke lokal | Mengambil hasil akhir agregasi ke laptop analis |
| `hdfs dfs -cat [path]` | Menampilkan isi file teks langsung | Mengintip 10 baris pertama file log HDFS |
| `hdfs dfs -du -h [path]` | Melihat penggunaan kapasitas disk | Memonitor kuota penyimpanan tiap folder |
| `hdfs dfs -count [path]` | Menghitung jumlah berkas & folder | Validasi jumlah partisi data |
| `hdfs dfs -rm [path]` | Menghapus file di HDFS | Membersihkan file sampah/sementara |

---
## 5. Latihan Mandiri

Selesaikan 4 latihan berikut untuk menguji pemahaman Anda terhadap manipulasi perintah HDFS.

**Soal 1.** Buatlah direktori baru di HDFS dengan path `/user/$USER/latihan3`.

In [ ]:
# Jawaban Soal 1: Membuat folder latihan3
!hdfs dfs -mkdir -p /user/$USER/latihan3
!hdfs dfs -ls /user/$USER

**Soal 2.** Buatlah berkas teks lokal bernama `biodata.txt` berisi biodata singkat kalian (nama, NIM, hobi — minimal 3 baris), lalu unggah berkas tersebut ke direktori `/user/$USER/latihan3` di HDFS.

In [ ]:
# Jawaban Soal 2: Buat teks biodata dan unggah ke HDFS
biodata = """Nama  : Sholahuddin Ahmad
NIM   : 250506070
Hobi  : Ngoprek Linux, Cloud Architecture, dan Big Data
"""

with open("biodata.txt", "w") as f:
    f.write(biodata)

# Upload ke direktori latihan3 di HDFS
!hdfs dfs -put -f biodata.txt /user/$USER/latihan3/
!hdfs dfs -ls /user/$USER/latihan3

**Soal 3.** Tampilkan isi berkas `biodata.txt` langsung dari HDFS menggunakan perintah `cat`.

In [ ]:
# Jawaban Soal 3: Menampilkan berkas dari HDFS
!hdfs dfs -cat /user/$USER/latihan3/biodata.txt

**Soal 4 (Refleksi Konsep).** Dalam 2-3 kalimat, jelaskan menurut pemahaman kalian: apa perbedaan mendasar antara menyimpan berkas di **disk lokal** dengan menyimpan di **HDFS**?

**Jawaban Refleksi:**
> Menyimpan berkas di **disk lokal** berarti data tersimpan secara utuh pada satu partisi drive penyimpanan fisik komputer tunggal tanpa replikasi otomatis. Jika drive tersebut mengalami kerusakan fisik (*hardware failure*), data akan hilang seketika.
>
> Sebaliknya, pada **HDFS**, berkas berukuran besar dipecah menjadi blok-blok terdistribusi (default 128 MB), disebarkan ke berbagai DataNode, dan direplikasi secara otomatis. Hal ini memberikan ketahanan tinggi terhadap kegagalan perangkat keras (*fault tolerance*) serta memungkinkan ribuan mesin memproses potongan-potongan blok data tersebut secara bersamaan (*parallel computation*).

---
## 6. TUGAS MANDIRI: Pipeline Data E-Commerce Multi-Cabang

### Skenario Bisnis:
Perusahaan tempat Anda bekerja memiliki tiga cabang kota (Magelang, Yogyakarta, dan Semarang). Sebagai Junior Data Analyst, Anda ditugaskan membangun pipeline data transaksi harian:
1. **Penyimpanan:** Membangun struktur folder penyimpanan terpisah antara data mentah (`raw`) dan data bersih (`processed`) di HDFS.
2. **Ingestion:** Mengunggah data CSV transaksi ketiga kota ke folder `raw` di HDFS.
3. **Streaming Reader:** Membaca data langsung dari stream HDFS ke DataFrame Pandas (tanpa menyentuh file lokal).
4. **Transformasi Bisnis:** Menggabungkan data, menghitung `total_pendapatan`, dan membuat ringkasan omzet per kota dan kategori.
5. **Output Sink:** Menyimpan hasil data olahan kembali ke zona `processed` di HDFS.

### Tahap Persiapan: Menghasilkan Tiga Dataset Cabang
Jalankan cell di bawah ini untuk membuat tiga berkas CSV simulasi (`transaksi_magelang.csv`, `transaksi_yogyakarta.csv`, `transaksi_semarang.csv`).

In [ ]:
import numpy as np
import pandas as pd

kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-08-01", "2026-08-31", freq="D")

cabang_kota = {"Magelang": 101, "Yogyakarta": 202, "Semarang": 303}

for kota, seed in cabang_kota.items():
    np.random.seed(seed)
    n = 200
    data_cabang = {
        "order_id": [f"{kota[:3].upper()}-{2000 + i}" for i in range(n)],
        "tanggal": np.random.choice(tanggal_range, size=n),
        "kategori": np.random.choice(kategori_list, size=n, p=[0.25, 0.25, 0.20, 0.15, 0.15]),
        "unit_terjual": np.random.randint(1, 8, size=n),
        "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000, 250000], size=n),
        "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    }
    df_cabang = pd.DataFrame(data_cabang)
    df_cabang["kota"] = kota
    nama_file = f"transaksi_{kota.lower()}.csv"
    df_cabang.to_csv(nama_file, index=False)
    print(f"Berkas '{nama_file}' berhasil dibuat ({df_cabang.shape[0]} baris)")

print("\nKetiga berkas CSV cabang siap digunakan untuk Tugas Mandiri.")

### Bagian A: Membangun Struktur Direktori HDFS (Bobot 15%)
Buat struktur folder berikut di HDFS:
```
/user/$USER/ecommerce/raw
/user/$USER/ecommerce/processed
```

In [ ]:
# Membuat direktori raw dan processed di HDFS
!hdfs dfs -mkdir -p /user/$USER/ecommerce/raw
!hdfs dfs -mkdir -p /user/$USER/ecommerce/processed

# Bukti direktori telah terbentuk
!hdfs dfs -ls /user/$USER/ecommerce

### Bagian B: Mengunggah Data Mentah ke HDFS (Bobot 15%)
Unggah ketiga berkas CSV transaksi cabang ke direktori `/user/$USER/ecommerce/raw/` di HDFS.

In [ ]:
# Upload 3 berkas CSV ke direktori HDFS raw
!hdfs dfs -put -f transaksi_magelang.csv /user/$USER/ecommerce/raw/
!hdfs dfs -put -f transaksi_yogyakarta.csv /user/$USER/ecommerce/raw/
!hdfs dfs -put -f transaksi_semarang.csv /user/$USER/ecommerce/raw/

# Tampilkan isi direktori beserta ukuran berkasnya
!hdfs dfs -ls -h /user/$USER/ecommerce/raw

### Bagian C: Membaca Kembali dan Menggabungkan Data dari HDFS (Bobot 25%)
**Instruksi Khusus Modul:** Mahasiswa diwajibkan membaca berkas **langsung dari HDFS** (bukan membaca berkas lokal).

Kita memanfaatkan fungsi `os.path.expandvars` agar jalur berkas bisa langsung menggunakan sintaks **`$USER`** tanpa perlu mendeklarasikan variabel terpisah.

In [ ]:
import io
import os
import subprocess
import pandas as pd

def baca_csv_langsung_dari_hdfs(hdfs_path):
    """Membaca isi berkas di HDFS secara streaming menggunakan stdout tanpa menyentuh disk lokal."""
    # Otomatis menerjemahkan $USER ke username aktif sistem
    path_asli = os.path.expandvars(hdfs_path)
    cmd = ["hdfs", "dfs", "-cat", path_asli]
    proses = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    stdout, stderr = proses.communicate()
    if proses.returncode != 0:
        raise RuntimeError(f"Gagal membaca file dari HDFS: {stderr}")
    return pd.read_csv(io.StringIO(stdout))

# Membaca ketiga file langsung dari HDFS menggunakan $USER secara langsung
df_magelang = baca_csv_langsung_dari_hdfs("/user/$USER/ecommerce/raw/transaksi_magelang.csv")
df_jogja    = baca_csv_langsung_dari_hdfs("/user/$USER/ecommerce/raw/transaksi_yogyakarta.csv")
df_semarang = baca_csv_langsung_dari_hdfs("/user/$USER/ecommerce/raw/transaksi_semarang.csv")

# Menggabungkan ketiga DataFrame menjadi satu
df_gabungan = pd.concat([df_magelang, df_jogja, df_semarang], ignore_index=True)

print(f"Total baris data gabungan: {len(df_gabungan)}")
print("\nDistribusi Transaksi per Cabang (value_counts):")
print(df_gabungan["kota"].value_counts())

# Tampilkan 5 baris teratas
df_gabungan.head()

### Bagian D: Mengolah dan Mengunggah Hasil ke Direktori `processed` (Bobot 25%)
1. Tambahkan kolom `total_pendapatan` (`unit_terjual * harga_satuan`).
2. Buat tabel ringkasan agregasi `groupby` total pendapatan per kota dan kategori.
3. Simpan `data_gabungan_bersih.csv` dan `ringkasan_kota_kategori.csv` ke lokal, lalu unggah keduanya ke direktori `/user/$USER/ecommerce/processed` di HDFS.

In [ ]:
# 1. Kalkulasi kolom total_pendapatan
df_gabungan["total_pendapatan"] = df_gabungan["unit_terjual"] * df_gabungan["harga_satuan"]

# 2. Agregasi pendapatan per kota dan kategori
df_ringkasan = (
    df_gabungan.groupby(["kota", "kategori"])["total_pendapatan"]
    .sum()
    .reset_index()
    .sort_values(by=["kota", "total_pendapatan"], ascending=[True, False])
)

print("=== RINGKASAN PENDAPATAN PER KOTA & KATEGORI ===")
display(df_ringkasan.head(10))

# 3. Simpan berkas hasil olahan ke disk lokal sementara
df_gabungan.to_csv("data_gabungan_bersih.csv", index=False)
df_ringkasan.to_csv("ringkasan_kota_kategori.csv", index=False)

# 4. Unggah kedua berkas hasil ke folder 'processed' di HDFS
!hdfs dfs -put -f data_gabungan_bersih.csv /user/$USER/ecommerce/processed/
!hdfs dfs -put -f ringkasan_kota_kategori.csv /user/$USER/ecommerce/processed/

# Verifikasi isi direktori processed
!hdfs dfs -ls -h /user/$USER/ecommerce/processed

### Bagian E: Dokumentasi dan Refleksi (Bobot 20%)

#### 1. Screenshot NameNode Web UI
Buka browser di **http://localhost:9870**, navigasikan ke menu **Utilities → Browse the file system**, lalu buka folder `/user/kaky/ecommerce/`.
Ambil screenshot tampilan browser yang membuktikan folder `raw` dan `processed` beserta file di dalamnya sudah ada di HDFS.

*(Sematkan gambar screenshot di bawah ini atau sertakan pada lampiran laporan)*

---

#### 2. Tulisan Reflektif (Minimal 100 Kata)
**Pertanyaan:** *Apa keuntungan menyimpan data mentah (raw) terpisah dari data olahan (processed) di HDFS, dibandingkan menyimpan semuanya bercampur dalam satu folder?*

**Jawaban Refleksi:**
> Memisahkan zona data mentah (*raw*) dan data olahan (*processed*) ke dalam direktori terpisah pada HDFS mengadopsi prinsip arsitektur data modern (seperti *Medallion Architecture*). Keuntungan fundamental utamanya adalah menjaga integritas data dan ketertelusuran riwayat pemrosesan (*data lineage*). Data mentah dipertahankan sebagai *single source of truth* yang bersifat *immutable* (tidak boleh dimodifikasi). Jika di kemudian hari ditemukan kesalahan pada algoritma pembersihan data atau terdapat perubahan formula kalkulasi bisnis, tim *data engineering* selalu dapat memproses ulang (*reprocess*) seluruh data dari kondisi awal tanpa risiko kehilangan data historis asli.
>
> Selain itu, pemisahan direktori mempermudah tata kelola akses (*access governance* dan *security*). Tim data analyst bisnis dapat langsung mengakses direktori *processed* yang sudah bersih dan optimal tanpa terbebani *noise* dari data mentah, sedangkan tim data engineer tetap dapat memantau kualitas data mentah yang baru masuk. Pemisahan ini juga mencegah terjadinya konflik penamaan berkas serta mempermudah penerapan retensi data (misalnya mengompres atau mengarsipkan data mentah lama secara otomatis).